# CairoGo - Part 0: ML Foundations (individual)
You are a data scientist in CairoGo's **driver team**. Two datasets in your Foundations pack:
- `drivers.csv` - one row per driver (with some mess). Targets: `churn_next_month` (0/1, will the driver leave next month?) and `next_month_earnings` (EGP).
- `telemetry.csv` - 30 sensor signals per ride. Target: `harsh_event` (0/1).

**Rules.** Fill only the cells marked `TODO`. Keep every `random_state` and every split exactly as given, so your numbers can be checked.
Each section matches one lab. Do each section after its lab. At the end the notebook writes `foundation_answers.json`.
Your version = last four digits of YOUR student ID mod 25. Use the pack `F_vXX` with that number.

In [2]:
STUDENT_ID = "20242187"          # <- your 8-digit ID
VERSION = int(STUDENT_ID[-4:]) % 25
PACK = f"F_v{VERSION:02d}"           # folder of your Foundations pack
import json, numpy as np, pandas as pd, warnings; warnings.filterwarnings("ignore")
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.compose import make_column_transformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, log_loss, precision_score, recall_score, f1_score, mean_absolute_error, mean_squared_error, r2_score, silhouette_score, adjusted_rand_score
ANS = {}
CV = StratifiedKFold(5, shuffle=True, random_state=42)
print("version", VERSION)

version 12


## 1. EDA and cleaning (Lab 1)
Look at the data first: types, describe(), missing values, duplicates, strange values, the target balance.

In [3]:
raw = pd.read_csv(f"{PACK}/drivers.csv")
print(raw.shape); print(raw.dtypes); print(raw.isna().mean().round(3))
# TODO: count exact duplicate rows in raw
ANS["n_duplicates"] = int(raw.duplicated().sum())  # your answer
# TODO: count impossible values of hours_online_week (a week has 168 hours)
ANS["n_impossible_hours"] = int((raw.hours_online_week > 168).sum())   # your answer
print(ANS)

(4896, 15)
driver_id                int64
age                    float64
tenure_months          float64
vehicle_type            object
home_zone               object
hours_online_week      float64
trips_last_30          float64
cash_share             float64
avg_rating             float64
cancellation_rate      float64
complaints_90d           int64
earnings_prev_month    float64
earnings_last_month    float64
churn_next_month         int64
next_month_earnings    float64
dtype: object
driver_id              0.000
age                    0.000
tenure_months          0.000
vehicle_type           0.000
home_zone              0.016
hours_online_week      0.000
trips_last_30          0.000
cash_share             0.000
avg_rating             0.084
cancellation_rate      0.030
complaints_90d         0.000
earnings_prev_month    0.000
earnings_last_month    0.000
churn_next_month       0.000
next_month_earnings    0.000
dtype: float64
{'n_duplicates': 96, 'n_impossible_hours': 25}


In [6]:
def clean(df):
    df = df.copy()
    # TODO: drop exact duplicates; set impossible hours_online_week (> 168) to NaN; strip spaces and lower-case vehicle_type
    pass   # your code here

    df = df.drop_duplicates()
    df.loc[df.hours_online_week > 168, "hours_online_week"] = np.nan
    df["vehicle_type"] = df["vehicle_type"].str.strip().str.lower()
    df["earnings_change"] = (df.earnings_last_month - df.earnings_prev_month) / df.earnings_prev_month.clip(lower=1)
    return df
df = clean(raw)
# TODO: share of missing avg_rating after cleaning, and churn rate

ANS["rating_missing_share"] = round(float(df.avg_rating.isna().mean()), 3)   # your answer
ANS["churn_rate"] = round(float(df.churn_next_month.mean()), 3)   # your answer
print(df.shape, ANS["rating_missing_share"], ANS["churn_rate"])

(4800, 16) 0.081 0.244


### Split and preprocessing (used by every section)
One split for everything: 80% train, 20% test, stratified on churn, `random_state=42`. All preprocessing is learned on **train only** (a Pipeline does that for you).

In [5]:
NUM = ["age", "tenure_months", "hours_online_week", "trips_last_30", "cash_share", "avg_rating", "cancellation_rate", "complaints_90d", "earnings_prev_month", "earnings_last_month", "earnings_change"]
CAT = ["vehicle_type", "home_zone"]
X = df[NUM + CAT]; y = df.churn_next_month.values; yr = df.next_month_earnings.values
Xtr, Xte, ytr, yte, yrtr, yrte = train_test_split(X, y, yr, test_size=0.2, random_state=42, stratify=y)
def pre(scale=True):
    # TODO: numeric -> median imputation (+ StandardScaler if scale); categorical -> most_frequent imputation + one-hot (handle_unknown="ignore", sparse_output=False)
    num_steps = [SimpleImputer(strategy="median")]
    if scale:
        num_steps.append(StandardScaler())
    num_pipe = make_pipeline(*num_steps)
    cat_pipe = make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OneHotEncoder(handle_unknown="ignore", sparse_output=False),
    )
    return make_column_transformer((num_pipe, NUM), (cat_pipe, CAT))
print(Xtr.shape, Xte.shape)

(3840, 13) (960, 13)


## 2. Linear regression (Lab 2) - target `next_month_earnings`
Fit the same model three ways: normal equation, gradient descent from scratch, scikit-learn.

In [7]:
P = pre().fit(Xtr); A = np.c_[np.ones(len(Xtr)), P.transform(Xtr)]; B = np.c_[np.ones(len(Xte)), P.transform(Xte)]
# TODO: normal equation theta = pinv(A^T A) A^T y ; test RMSE
theta = np.linalg.pinv(A.T @ A) @ A.T @ yrtr
pred_ne = B @ theta
ANS["rmse_normal_eq"] = round(float(np.sqrt(mean_squared_error(yrte, pred_ne))), 3)   # your answer
# TODO: batch gradient descent on MSE, theta starts at zeros, learning rate 0.1, 3000 iterations; test RMSE
theta_gd = np.zeros(A.shape[1])
n = len(A)
for _ in range(3000):
    grad = (2 / n) * A.T @ (A @ theta_gd - yrtr)
    theta_gd -= 0.1 * grad
pred_gd = B @ theta_gd
ANS["rmse_gd"] = round(float(np.sqrt(mean_squared_error(yrte, pred_gd))), 3)   # your answer
from sklearn.linear_model import LinearRegression
lr = make_pipeline(pre(), LinearRegression()).fit(Xtr, yrtr); pr = lr.predict(Xte)
# TODO: test MAE and R2 of the scikit-learn model
ANS["mae_linreg"] = round(float(mean_absolute_error(yrte, pr)), 3)   # your answer
ANS["r2_linreg"] = round(float(r2_score(yrte, pr)), 3)   # your answer
print({k: ANS[k] for k in ("rmse_normal_eq", "rmse_gd", "mae_linreg", "r2_linreg")})

{'rmse_normal_eq': 1084.378, 'rmse_gd': 1084.358, 'mae_linreg': 864.507, 'r2_linreg': 0.814}


## 3. Logistic regression (Lab 3) - target `churn_next_month`
Metrics: ROC-AUC, log loss (= cross entropy), precision and recall at 0.5. Then choose a threshold on a **validation** split, never on test.

In [8]:
from sklearn.linear_model import LogisticRegression
logm = make_pipeline(pre(), LogisticRegression(max_iter=1000)).fit(Xtr, ytr); p = logm.predict_proba(Xte)[:, 1]
# TODO: test AUC, test log loss, precision and recall with threshold 0.5
ANS["auc_logistic"] = round(float(roc_auc_score(yte, p)), 3)   # your answer
ANS["logloss_logistic"] = round(float(log_loss(yte, p)), 3)   # your answer
ANS["precision_logistic"] = round(float(precision_score(yte, p >= 0.5)), 3)   # your answer
ANS["recall_logistic"] = round(float(recall_score(yte, p >= 0.5)), 3)  # your answer
Xa, Xv, ya, yv = train_test_split(Xtr, ytr, test_size=0.25, random_state=42, stratify=ytr)
pv = make_pipeline(pre(), LogisticRegression(max_iter=1000)).fit(Xa, ya).predict_proba(Xv)[:, 1]
# TODO: threshold in 0.05, 0.10, ..., 0.95 with the best F1 on the validation split
ths = np.round(np.arange(0.05, 0.96, 0.05), 2)
f1s = [f1_score(yv, pv >= t) for t in ths]
ANS["best_threshold_f1"] = float(ths[int(np.argmax(f1s))])   # your answer
print({k: ANS[k] for k in ANS if "logistic" in k or "threshold" in k})

{'auc_logistic': 0.775, 'logloss_logistic': 0.456, 'precision_logistic': 0.664, 'recall_logistic': 0.338, 'best_threshold_f1': 0.3}


## 4. kNN and Naive Bayes (Lab 4)
Choose k with 5-fold cross-validation on train (scoring = roc_auc). Then see what happens to kNN without scaling.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
# TODO: GridSearchCV over k in [1, 5, 11, 21, 41, 81] with cv=CV, scoring="roc_auc"
ANS["best_k"] = None   # your answer
k = ANS["best_k"]
auc_s = roc_auc_score(yte, make_pipeline(pre(True), KNeighborsClassifier(k)).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
auc_u = roc_auc_score(yte, make_pipeline(pre(False), KNeighborsClassifier(k)).fit(Xtr, ytr).predict_proba(Xte)[:, 1])
# TODO: is kNN better WITH scaling? answer "yes" or "no"; and Gaussian NB test AUC
ANS["knn_scaling_helps"] = None   # your answer
ANS["auc_nb"] = None   # your answer
print(round(auc_s, 3), round(auc_u, 3), ANS["knn_scaling_helps"], ANS["auc_nb"])

## 5. Decision trees (Lab 5)
Choose `max_depth` with the same 5-fold CV. Trees do not need scaling; why?

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# TODO: GridSearchCV over max_depth in [2, 3, 4, 5, 6, 8, 12] (random_state=0), cv=CV, scoring="roc_auc"; test AUC of the best tree
ANS["best_depth"] = None   # your answer
ANS["auc_tree"] = None   # your answer
print(ANS["best_depth"], ANS["auc_tree"])

## 6. SVM (Lab 6)
Grid search C and gamma inside a pipeline. To keep it fast, use the first 2000 training rows.

In [ ]:
from sklearn.svm import SVC
# TODO: GridSearchCV over C in [0.3, 1, 3] and gamma in ["scale", 0.03, 0.3] on Xtr.iloc[:2000], cv=CV, scoring="roc_auc"; test AUC (use decision_function)
ANS["svm_best_C"] = None   # your answer
ANS["auc_svm"] = None   # your answer
print(ANS["svm_best_C"], ANS["svm_best_gamma"], ANS["auc_svm"])

## 7. Neural network from scratch (Lab 7) - `telemetry.csv`
A 2-layer network: 16 tanh hidden units, sigmoid output, cross-entropy loss, full-batch gradient descent. Use the first 3000 rows.

In [ ]:
tel = pd.read_csv(f"{PACK}/telemetry.csv"); S = [c for c in tel.columns if c.startswith("s")]
T = tel[S].values; ty = tel.harsh_event.values
Ttr, Tte, tytr, tyte = train_test_split(T, ty, test_size=0.25, random_state=42, stratify=ty)
ts = StandardScaler().fit(Ttr); Za, Zb = ts.transform(Ttr), ts.transform(Tte)
rng = np.random.default_rng(0); W1 = rng.normal(0, 0.3, (Za.shape[1], 16)); b1 = np.zeros(16); W2 = rng.normal(0, 0.3, 16); b2 = 0.0
Xs, ys = Za[:3000], tytr[:3000]
for epoch in range(500):
    # TODO: forward pass (tanh hidden, sigmoid output), then backprop of the mean cross-entropy, update with learning rate 0.5
    pass   # your code here
o = 1 / (1 + np.exp(-(np.tanh(Xs @ W1 + b1) @ W2 + b2)))
ANS["scratch_train_cross_entropy"] = round(float(log_loss(ys, o)), 3)
print(ANS["scratch_train_cross_entropy"])

## 8. Neural networks in practice (Lab 8)
`MLPClassifier` here so the deterministic numbers can be checked. Then run the required Keras practice cell below in Colab; its exact numeric output is not auto-keyed.
(a) NN vs the other models on telemetry and on drivers. (b) NN without scaling. (c) A too-big network on little data.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
mlp = lambda: MLPClassifier((64, 32), early_stopping=True, max_iter=500, random_state=0)
# TODO: telemetry test AUC of: MLP (scaled), logistic (scaled), HistGradientBoosting (random_state=0), SVC (scaled, decision_function), MLP WITHOUT scaling
pass   # your code here
ANS["auc_mlp_telemetry"] = round(float(a_mlp), 3)
ANS["nn_best_on_telemetry"] = "yes" if a_mlp > max(a_log, a_hgb, a_svm) else "no"
ANS["unscaled_nn_drop_over_0.1"] = "yes" if a_mlp - a_raw > 0.1 else "no"
# TODO: drivers: MLP test AUC (with pre()); is the NN better than your best tree from section 5? "yes"/"no"
ANS["auc_mlp_drivers"] = None   # your answer
ANS["nn_beats_tree_on_drivers"] = None   # your answer
big = MLPClassifier((256, 128, 64), alpha=0.0, max_iter=300, random_state=0).fit(Za[:1500], tytr[:1500])
small = MLPClassifier((32, 16), alpha=3.0, max_iter=500, random_state=0).fit(Za[:1500], tytr[:1500])   # smaller + strong L2
# TODO: big network: train AUC minus test AUC > 0.1? ("yes"/"no"); does the small regularised network get a higher TEST AUC than the big one? ("yes"/"no")
ANS["big_nn_overfits"] = None   # your answer
ANS["small_nn_generalises_better"] = None   # your answer
print({k: ANS[k] for k in ANS if "nn" in k or "mlp" in k})

### Required Keras practice (Lab 8)
Keras is part of the course, so run this cell in **Google Colab** after you finish the scikit-learn comparison. The exact Keras number is deliberately **not** part of the deterministic answer key because hosted notebook images and deep-learning libraries change more often than the rest of the grader. Your job is to run it, inspect train/validation behaviour, and compare it with the MLP result above.

Questions to answer in a markdown cell below the output: (1) Did validation improve after early stopping? (2) Did Keras change the conclusion about whether a neural network is the best choice on telemetry?

In [ ]:
# Keras learning cell - run in Colab. This cell is required learning material, but its exact numeric output is not auto-keyed.
try:
    import tensorflow as tf
    tf.keras.utils.set_random_seed(0)
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(Za.shape[1],)),
        tf.keras.layers.Dense(64, activation="relu"),
        tf.keras.layers.Dense(32, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ])
    model.compile(optimizer="adam", loss="binary_crossentropy", metrics=[tf.keras.metrics.AUC(name="auc")])
    cb = tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
    hist = model.fit(Za, tytr, validation_split=0.2, epochs=60, batch_size=64, callbacks=[cb], verbose=0)
    keras_auc = roc_auc_score(tyte, model.predict(Zb, verbose=0).ravel())
    print("Keras test AUC:", round(float(keras_auc), 3), "epochs:", len(hist.history["loss"]))
except Exception as e:
    print("Keras cell not executed in this environment. Run it in Google Colab. Reason:", type(e).__name__, str(e)[:120])

## 9. Clustering (Lab 9) - driver profiles
Features: hours_online_week, trips_last_30, tenure_months, cash_share (median-imputed, scaled).

In [ ]:
from sklearn.cluster import KMeans, AgglomerativeClustering
F = df[["hours_online_week", "trips_last_30", "tenure_months", "cash_share"]]
Fz = StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(F))
# TODO: K-means (n_init=10, random_state=42) for k = 2..8; k with the best silhouette; agglomerative (ward) at that k; adjusted Rand index between the two partitions
ANS["best_k_silhouette"] = None   # your answer
ANS["ari_kmeans_vs_agglomerative"] = None   # your answer
print(ANS["best_k_silhouette"], ANS["ari_kmeans_vs_agglomerative"])

## Save your answers
Upload this notebook and `foundation_answers.json` inside `I_<yourID>.zip`.

In [ ]:
json.dump({"student_id": STUDENT_ID, "version": VERSION, "answers": ANS}, open("foundation_answers.json", "w"), indent=1)
print(len(ANS), "answers saved")